In [ ]:
from src import WhatsappClient

In [ ]:
client = WhatsappClient(DEBUG=True)
await client.initialize_playwright()

In [ ]:
await client.login()

In [ ]:
await client.extract_chat_details_from_side_pane()
# await client.search_pane_scroll_down()

In [ ]:
await client.fetch_latest_message()

In [ ]:
await client.on_new_message(lambda x: print(x))

In [ ]:
# await client.search_pane_scroll_down()
await client.chat_pane_scroll_up()

In [ ]:
await client.open_chat_panel("Prathwik")

In [ ]:
from tabulate import tabulate

messages = await client.extract_messages()

table_data = [
    [msg["sender"], msg["time"], msg["message"], str(msg["attachment"])]
    for msg in messages
]
headers = ["Sender", "Time Sent", "Message", "Attachment Details"]

print("Number of previous messages: ", len(messages))
print(tabulate(table_data, headers=headers, tablefmt="grid"))

In [7]:
## LEAVE THIS AS IT IS GOOD FOR REFACTORING STUFF
import asyncio
from playwright.async_api import async_playwright
import os

BASE_URL = "https://web.whatsapp.com"


async def initialize_playwright(browser_instance, user_data_dir, headless):
    # TODO: Perform browser level optimizations and other stuff
    playwright = await async_playwright().start()

    browser = await playwright[browser_instance].launch_persistent_context(
        user_data_dir, headless=headless
    )

    page_instance = await browser.new_page()

    # await page_instance.set_viewport_size({"width": 1920, "height": 1080})
    return playwright, browser, page_instance


async def login(page, user_data_dir):
    # TODO: QR code & phone number login via script

    await page.goto(BASE_URL)
    await page.bring_to_front()

    print("Waiting for WhatsApp chats to load...")
    await page.wait_for_selector(
        '//*[@id="pane-side"]/div[2]/div/div/child::div', timeout=600000
    )
    print("WhatsApp chats loaded.")

In [8]:
playwright, browser, page = await initialize_playwright("chromium", "user_data", False)
await login(page, "user_data")

Waiting for WhatsApp chats to load...
WhatsApp chats loaded.


In [30]:
## ADD TESTING STUFF HERE
import time


async def chat_pane_scroll_up(self, scroll_duration=10, scroll_delta=1000, delay=0.1):
    """
    Scrolls up the chat pane (div with id="main") by moving the mouse cursor
    to the center of the pane and performing rapid scroll wheel actions to reach the top.
    """

    scroll_selector = "#main"
    chat_panel_selector = "div[id='main']"

    try:
        chat_panel = await self.page.query_selector(chat_panel_selector)
        if not chat_panel:
            logger.error("Could not locate the chat pane for scrolling.")
            return

        pane = self.page.locator(scroll_selector)
        bounding_box = await pane.bounding_box()
        if not bounding_box:
            logger.error("Could not locate the chat pane for scrolling.")
            return

        center_x = bounding_box["x"] + bounding_box["width"] / 2
        center_y = bounding_box["y"] + bounding_box["height"] / 2

        logger.info(
            "Starting to scroll up the chat pane to reach the top by rapid mouse scrolling."
        )

        end_time = time.time() + scroll_duration
        while time.time() < end_time:
            await self.page.mouse.move(center_x, center_y)
            await self.page.mouse.wheel(0, -scroll_delta)  # Increased scroll up delta
            await asyncio.sleep(delay)  # Reduced sleep time for faster scrolling

    except Exception as e:
        logger.exception(f"Error while scrolling up the chat pane using mouse: {e}")
        return

In [31]:
await chat_pane_scroll_up()